# heatmap-figure

模板 / Template: `heatmap`  

修改 `PLOT_SPEC` 以调整数据语义，修改 `CHART_STYLE` 以调整外观。每次改变科学含义后，请从新 kernel 运行全部单元。

In [ ]:
from pathlib import Path
import hashlib
import json
import os
import platform
import tempfile

import matplotlib
import matplotlib.pyplot as plt
from matplotlib.ticker import StrMethodFormatter
import numpy as np
import pandas as pd

NOTEBOOK_DIR = Path.cwd()
if NOTEBOOK_DIR.name != "notebooks":
    raise RuntimeError("Run this notebook with notebooks/ as the working directory")
WORKSPACE_ROOT = NOTEBOOK_DIR.parent
SOURCE_DIR = WORKSPACE_ROOT / "data" / "source"
FIGURE_DIR = WORKSPACE_ROOT / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
# figure-style：共享 base.mplstyle 的自包含副本。
# figure-style: self-contained copy of the shared base.mplstyle.
# 修改带注释的文本后，请重新运行本单元和绘图单元。
# Edit the annotated text, then rerun this cell and the plotting cells.
BASE_MPLSTYLE = r"""# 科研绘图基础样式 / Base style for scientific figures

# 画布 / Canvas
figure.figsize: 3.35, 2.70
figure.dpi: 100
figure.facecolor: white
figure.constrained_layout.use: True

# 字体排印 / Typography
font.family: sans-serif
font.sans-serif: Arial, Liberation Sans, DejaVu Sans
font.size: 8.5
axes.titlesize: 9.5
axes.labelsize: 10
axes.labelweight: bold

# 坐标轴 / Axes
axes.facecolor: white
axes.edgecolor: 8C8C8C
axes.linewidth: 0.65
axes.axisbelow: True
axes.grid: True
axes.spines.top: True
axes.spines.right: True

# 系列循环：颜色、marker 和线型列表必须等长 / Series cycle: lists must have equal length
# marker：v 下三角，o 圆，s 方块，^ 上三角，D 菱形，X 实心叉
# markers: v down-triangle, o circle, s square, ^ up-triangle, D diamond, X filled-X
axes.prop_cycle: cycler(color=['0C84C6', '41B7AC', 'FFA510', 'F74D4D', '2455A4', '002C53']) + cycler(marker=['v', 'o', 's', '^', 'D', 'X']) + cycler(linestyle=['-', '--', '-.', ':', '-', '--'])

# 刻度 / Ticks
xtick.labelsize: 8.5
ytick.labelsize: 8.5
xtick.major.size: 2
ytick.major.size: 2
xtick.major.width: 0.6
ytick.major.width: 0.6

# 线、marker 与色块 / Lines, markers, and patches
lines.linewidth: 1.3
lines.markersize: 4.2
patch.linewidth: 0.65
patch.edgecolor: white
patch.force_edgecolor: True

# 图例 / Legend
legend.fontsize: 8.5
legend.frameon: True
legend.fancybox: True
legend.framealpha: 0.96
legend.edgecolor: A6A6A6
legend.facecolor: white
legend.handlelength: 1.7
legend.handletextpad: 0.5
legend.borderpad: 0.45

# 网格 / Grid
grid.color: 9A9A9A
grid.linestyle: --
grid.linewidth: 0.5
grid.alpha: 0.5

# 导出 / Export
savefig.dpi: 300
savefig.facecolor: white
savefig.transparent: False
pdf.fonttype: 42
ps.fonttype: 42
svg.fonttype: none"""

def _rcparams_from_mplstyle_text(text):
    """解析原生 Matplotlib key，同时忽略空行和注释行。

    Parse native Matplotlib keys while ignoring blank/comment lines.
    """
    params = {}
    for line_number, raw_line in enumerate(text.splitlines(), start=1):
        line = raw_line.strip()
        if not line or line.startswith("#"):
            continue
        if ":" not in raw_line:
            raise ValueError(f"Style line {line_number} must use 'key: value'")
        key, value = (part.strip() for part in raw_line.split(":", 1))
        if key not in matplotlib.rcParams.validate:
            raise KeyError(f"Unknown Matplotlib rcParam on style line {line_number}: {key}")
        params[key] = matplotlib.rcParams.validate[key](value)
    return params

BASE_STYLE_SOURCE = {'filename': 'base.mplstyle',
 'sha256': 'cc3e628ba81f2b83136580d00247d2c56f22547348a3e113e4e788cea75a8b93'}
PAPER_STYLE = _rcparams_from_mplstyle_text(BASE_MPLSTYLE)
_STYLE_CYCLE = PAPER_STYLE["axes.prop_cycle"].by_key()
STYLE_COLORS = _STYLE_CYCLE["color"]
STYLE_MARKERS = _STYLE_CYCLE.get("marker", ["o"] * len(STYLE_COLORS))
STYLE_LINESTYLES = _STYLE_CYCLE.get("linestyle", ["-"] * len(STYLE_COLORS))


In [ ]:
FIGURE_METADATA = {'schema_version': '1.0',
 'paper_name': 'Paper name',
 'figure_slug': 'heatmap-figure',
 'chart_type': 'heatmap',
 'claim': 'Replace with the precise claim this figure supports.',
 'data_sources': [{'workspace_path': 'data/source/results.csv', 'sha256': 'computed-at-runtime'}],
 'transformations': ['Required columns only; numeric value conversion; duplicate cells are '
                     'rejected rather than aggregated.'],
 'missing_values': {'policy': 'reject', 'reason': 'Missing values are never dropped silently.'},
 'uncertainty': {'type': 'none', 'source': None},
 'axis_policy': {'x_scale': 'categorical',
                 'y_scale': 'categorical',
                 'color_scale': 'configured colormap and bounds'},
 'dimensions_inches': [3.35, 2.4],
 'palette': {'provider': 'matplotlib-colormap',
             'source_kind': 'continuous-colormap',
             'id': 'viridis',
             'source_url': 'https://matplotlib.org/stable/users/explain/colors/colormaps.html',
             'colors': ['#440154', '#21918C', '#FDE725'],
             'base_style': {'filename': 'base.mplstyle',
                            'source_sha256': 'cc3e628ba81f2b83136580d00247d2c56f22547348a3e113e4e788cea75a8b93',
                            'embedded_sha256': 'computed-at-runtime'}},
 'outputs': ['heatmap-figure.pdf', 'heatmap-figure.svg', 'heatmap-figure.png']}


## 使用说明 / How to use

1. `FIGURE_METADATA` 记录科学含义与数据处理；`PLOT_SPEC` 只映射数据列。
2. `CHART_STYLE` 只控制画布、图形标记、图例和坐标轴，不应改变统计含义。
3. 模板不会生成或补齐实验数据；缺失值和重复观测必须显式处理。
4. 修改后从新 kernel 运行全部单元，并检查 PDF、SVG 与至少 300 PPI 的 PNG。

In [ ]:
SOURCE_FILE = SOURCE_DIR / "results.csv"
if not SOURCE_FILE.is_file():
    raise FileNotFoundError(SOURCE_FILE)

def file_sha256(path):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

suffix = SOURCE_FILE.suffix.lower()
if suffix in {".csv", ".txt"}:
    source_data = pd.read_csv(SOURCE_FILE)
elif suffix == ".tsv":
    source_data = pd.read_csv(SOURCE_FILE, sep="\t")
elif suffix in {".xlsx", ".xls"}:
    source_data = pd.read_excel(SOURCE_FILE)
elif suffix == ".json":
    source_data = pd.read_json(SOURCE_FILE)
else:
    raise ValueError(f"Unsupported source format: {suffix}")
FIGURE_METADATA["data_sources"][0]["sha256"] = file_sha256(SOURCE_FILE)
source_data.head()


In [ ]:
# 修改这个映射后，请重新运行全部单元。
# Edit this mapping, then rerun all cells.
PLOT_SPEC = {'x_column': 'column',
 'y_column': 'row',
 'value_column': 'value',
 'x_label': 'Column',
 'y_label': 'Row',
 'colorbar_label': 'Value'}

In [ ]:
# chart-style：沿用折线图和柱状图的完整框线、网格、字体与图例规范。
# Match the closed-frame line/bar convention while preserving chart semantics.
CHART_STYLE = {'canvas': {'figure_size': [3.35, 2.4],
            'figure_facecolor': 'white',
            'axes_facecolor': 'white',
            'grid': {'show': False,
                     'axis': 'both',
                     'color': '#9A9A9A',
                     'linestyle': '--',
                     'linewidth': 0.5,
                     'alpha': 0.5},
            'spines': {'visible': ['left', 'right', 'top', 'bottom'],
                       'color': '#8C8C8C',
                       'linewidth': 0.65,
                       'alpha': 0.8}},
 'marks': {'colors': ['#440154', '#21918C', '#FDE725'],
           'cmap': 'viridis',
           'vmin': None,
           'vmax': None,
           'center': None,
           'aspect': 'auto',
           'interpolation': 'nearest',
           'missing_color': '#D9D9D9',
           'annotate': True,
           'annotation_format': '.2g',
           'annotation_color': 'auto',
           'annotation_fontsize': 7,
           'colorbar': True,
           'colorbar_fraction': 0.046,
           'colorbar_pad': 0.04},
 'legend': {'show': False,
            'loc': 'best',
            'bbox_to_anchor': None,
            'ncol': 1,
            'fontsize': 8.5,
            'title': None,
            'title_fontsize': 8.5,
            'facecolor': 'white',
            'edgecolor': '#A6A6A6',
            'framealpha': 0.96,
            'linewidth': 0.65},
 'axes': {'label': {'fontsize': 10.0, 'fontweight': 'bold', 'color': '#222222'},
          'x': {'limits': None,
                'tick_fontsize': 8.5,
                'tick_fontweight': 'bold',
                'tick_rotation': 25,
                'tick_format': None},
          'y': {'limits': None,
                'tick_fontsize': 8.5,
                'tick_fontweight': 'bold',
                'tick_rotation': 0,
                'tick_format': None}}}

In [ ]:
required = [PLOT_SPEC["x_column"], PLOT_SPEC["y_column"], PLOT_SPEC["value_column"]]
missing_columns = [name for name in required if name not in source_data.columns]
if missing_columns:
    raise KeyError(f"Missing required columns: {missing_columns}")
if source_data[required].isna().any().any():
    raise ValueError("Missing values found. Declare and implement a policy before plotting.")

plot_data = source_data[required].copy()
plot_data[PLOT_SPEC["value_column"]] = pd.to_numeric(
    plot_data[PLOT_SPEC["value_column"]], errors="raise"
)
duplicate_cells = plot_data.duplicated([PLOT_SPEC["x_column"], PLOT_SPEC["y_column"]], keep=False)
if duplicate_cells.any():
    raise ValueError("Duplicate heatmap cells found. Aggregate explicitly and record the method first.")

x_order = list(dict.fromkeys(plot_data[PLOT_SPEC["x_column"]].tolist()))
y_order = list(dict.fromkeys(plot_data[PLOT_SPEC["y_column"]].tolist()))
matrix = (
    plot_data.pivot(index=PLOT_SPEC["y_column"], columns=PLOT_SPEC["x_column"], values=PLOT_SPEC["value_column"])
    .reindex(index=y_order, columns=x_order)
    .to_numpy(dtype=float)
)
masked_matrix = np.ma.masked_invalid(matrix)

In [ ]:
from matplotlib.colors import TwoSlopeNorm

def _style_axes(ax):
    canvas, axes = CHART_STYLE["canvas"], CHART_STYLE["axes"]
    ax.set_facecolor(canvas["axes_facecolor"])
    grid = canvas["grid"]
    if grid["show"]:
        ax.grid(
            True, axis=grid["axis"], color=grid["color"],
            linestyle=grid["linestyle"], linewidth=grid["linewidth"], alpha=grid["alpha"],
        )
    else:
        ax.grid(False)
    spines = canvas["spines"]
    for name, spine in ax.spines.items():
        spine.set_visible(name in spines["visible"])
        spine.set_color(spines["color"])
        spine.set_linewidth(spines["linewidth"])
        spine.set_alpha(spines["alpha"])
    for axis_name in ("x", "y"):
        cfg = axes[axis_name]
        if "scale" in cfg:
            getattr(ax, f"set_{axis_name}scale")(cfg["scale"])
        if cfg["limits"] is not None:
            getattr(ax, f"set_{axis_name}lim")(*cfg["limits"])
        if cfg["tick_format"] is not None:
            getattr(ax, f"{axis_name}axis").set_major_formatter(StrMethodFormatter(cfg["tick_format"]))
        ax.tick_params(axis=axis_name, labelsize=cfg["tick_fontsize"])
        for label in getattr(ax, f"get_{axis_name}ticklabels")():
            label.set_fontweight(cfg["tick_fontweight"])
            label.set_rotation(cfg["tick_rotation"])
            if axis_name == "x" and cfg["tick_rotation"]:
                label.set_horizontalalignment("right")
    label = axes["label"]
    ax.set_xlabel(PLOT_SPEC["x_label"], **label)
    ax.set_ylabel(PLOT_SPEC["y_label"], **label)


def _add_legend(ax, handles=None):
    cfg = CHART_STYLE["legend"]
    if not cfg["show"]:
        return None
    kwargs = {
        "loc": cfg["loc"], "ncol": cfg["ncol"], "fontsize": cfg["fontsize"],
        "title": cfg["title"], "title_fontsize": cfg["title_fontsize"], "frameon": True,
    }
    if cfg["bbox_to_anchor"] is not None:
        kwargs["bbox_to_anchor"] = cfg["bbox_to_anchor"]
    if handles is not None:
        kwargs["handles"] = handles
    item = ax.legend(**kwargs)
    frame = item.get_frame()
    frame.set_facecolor(cfg["facecolor"])
    frame.set_edgecolor(cfg["edgecolor"])
    frame.set_alpha(cfg["framealpha"])
    frame.set_linewidth(cfg["linewidth"])
    return item


def build_figure():
    with plt.rc_context(PAPER_STYLE):
        canvas, marks = CHART_STYLE["canvas"], CHART_STYLE["marks"]
        fig, ax = plt.subplots(figsize=canvas["figure_size"])
        fig.patch.set_facecolor(canvas["figure_facecolor"])
        cmap = matplotlib.colormaps[marks["cmap"]].copy()
        cmap.set_bad(marks["missing_color"])
        options = {
            "cmap": cmap, "aspect": marks["aspect"], "interpolation": marks["interpolation"],
            "vmin": marks["vmin"], "vmax": marks["vmax"],
        }
        if marks["center"] is not None:
            low = marks["vmin"] if marks["vmin"] is not None else float(np.nanmin(matrix))
            high = marks["vmax"] if marks["vmax"] is not None else float(np.nanmax(matrix))
            if not low < marks["center"] < high:
                raise ValueError("marks.center must lie strictly between the displayed minimum and maximum.")
            options.pop("vmin")
            options.pop("vmax")
            options["norm"] = TwoSlopeNorm(vmin=low, vcenter=marks["center"], vmax=high)
        image = ax.imshow(masked_matrix, **options)
        ax.set_xticks(np.arange(len(x_order)), [str(item) for item in x_order])
        ax.set_yticks(np.arange(len(y_order)), [str(item) for item in y_order])
        _style_axes(ax)
        if marks["annotate"]:
            for row in range(matrix.shape[0]):
                for column in range(matrix.shape[1]):
                    value = matrix[row, column]
                    if np.isfinite(value):
                        rgba = image.cmap(image.norm(value))
                        luminance = 0.2126 * rgba[0] + 0.7152 * rgba[1] + 0.0722 * rgba[2]
                        color = (
                            "#111111" if marks["annotation_color"] == "auto" and luminance > 0.58
                            else "white" if marks["annotation_color"] == "auto"
                            else marks["annotation_color"]
                        )
                        ax.text(
                            column, row, format(value, marks["annotation_format"]),
                            ha="center", va="center", color=color,
                            fontsize=marks["annotation_fontsize"],
                        )
        if marks["colorbar"]:
            colorbar = fig.colorbar(
                image, ax=ax, fraction=marks["colorbar_fraction"], pad=marks["colorbar_pad"]
            )
            label = CHART_STYLE["axes"]["label"]
            colorbar.set_label(PLOT_SPEC["colorbar_label"], **label)
            colorbar.ax.tick_params(labelsize=CHART_STYLE["axes"]["y"]["tick_fontsize"])
            for item in colorbar.ax.get_yticklabels():
                item.set_fontweight(CHART_STYLE["axes"]["y"]["tick_fontweight"])
        return fig

In [ ]:
def atomic_save_figure(figure, destination, dpi=300):
    destination = Path(destination)
    descriptor, temporary_name = tempfile.mkstemp(prefix=f".{destination.stem}.", suffix=destination.suffix, dir=destination.parent)
    os.close(descriptor)
    try:
        options = {"format": destination.suffix.lstrip("."), "bbox_inches": None, "facecolor": CHART_STYLE["canvas"]["figure_facecolor"]}
        if destination.suffix.lower() == ".png": options["dpi"] = dpi
        figure.savefig(temporary_name, **options); os.replace(temporary_name, destination)
    except Exception:
        Path(temporary_name).unlink(missing_ok=True); raise

FIGURE_METADATA["dimensions_inches"] = list(CHART_STYLE["canvas"]["figure_size"])
FIGURE_METADATA["palette"]["colors"] = list(CHART_STYLE["marks"]["colors"])
FIGURE_METADATA["palette"]["base_style"]["embedded_sha256"] = hashlib.sha256(BASE_MPLSTYLE.encode("utf-8")).hexdigest()
figure = build_figure()
for output_name in FIGURE_METADATA["outputs"]: atomic_save_figure(figure, FIGURE_DIR / output_name)
manifest = dict(FIGURE_METADATA)
manifest["environment"] = {"python": platform.python_version(), "matplotlib": matplotlib.__version__, "pandas": pd.__version__}
manifest_path = FIGURE_DIR / f"{FIGURE_METADATA['figure_slug']}.manifest.json"
descriptor, temporary_name = tempfile.mkstemp(prefix=f".{manifest_path.stem}.", suffix=manifest_path.suffix, dir=manifest_path.parent); os.close(descriptor)
try:
    Path(temporary_name).write_text(json.dumps(manifest, indent=2, ensure_ascii=False) + "\n", encoding="utf-8"); os.replace(temporary_name, manifest_path)
except Exception:
    Path(temporary_name).unlink(missing_ok=True); raise
figure
